In [30]:
import os
import numpy as np
import torch
import open3d as o3d
from scipy.spatial import KDTree
import plotly.graph_objects as go

ROOT = os.path.abspath('')
EXP  = os.path.join(ROOT, 'experiments',
                    'geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn')
TRAIN_DIR = os.path.join(ROOT, 'data', 'faces', 'data', 'train')

# Load reference (average face — same for all training pairs)
ref_raw = torch.load(
    os.path.join(TRAIN_DIR, '3478', 'full_face.pth'), weights_only=False)
ref_pts = (ref_raw.numpy() if isinstance(ref_raw, torch.Tensor)
           else np.array(ref_raw)).astype(np.float32)[:, :3]

# Real scans
SCANS = {
    'plank_scaled': np.load(os.path.join(EXP, 'plank_scaled.npy'))[:, :3].astype(np.float32),
    '2189':         np.load(os.path.join(EXP, '2189.npy'))[:, :3].astype(np.float32),
}

def rms(pts):
    c = pts.mean(0)
    return float(np.sqrt(np.mean(np.sum((pts - c)**2, axis=1))))

print(f'ref_pts : {ref_pts.shape}   RMS={rms(ref_pts):.4f}')
for name, pts in SCANS.items():
    print(f'{name:<16}: {pts.shape}   RMS={rms(pts):.4f}')

ref_pts : (10788, 3)   RMS=0.7781
plank_scaled    : (2000, 3)   RMS=0.5698
2189            : (5660, 3)   RMS=0.3561


In [31]:
def umeyama(src, dst):
    """Closed-form s, R, t minimising ||s*R@src.T + t[:,None] - dst.T||^2."""
    n = len(src)
    src_m, dst_m = src.mean(0), dst.mean(0)
    src_c, dst_c = src - src_m, dst - dst_m
    cov = (dst_c.T @ src_c) / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3, dtype=np.float64)
    S[2, 2] = np.linalg.det(U @ Vt)   # handle reflection
    R = U @ S @ Vt
    src_var = float(np.sum(src_c**2) / n)
    s = float(np.trace(np.diag(D) @ S) / src_var)
    t = dst_m - s * R @ src_m
    return s, R, t

In [32]:
def scale_icp(src, ref, n_iter=20, dist_thresh=0.25, thresh_decay=0.90,
              min_inliers=50, init_R=None, init_t=None, icp_ref=None,
              center='centroid'):
    """
    Estimate scale s such that s*R@src + t ≈ ref.
    center: 'centroid' (default) or 'nose' — anchor src's max-Z point to ref's max-Z point.
    icp_ref: subset of ref used for NN search (e.g. front-face only).
    Returns (s_total, aligned_src).
    """
    src = src.astype(np.float64)
    ref = ref.astype(np.float64)

    if init_R is not None:
        src = (init_R @ src.T).T + (init_t if init_t is not None else 0)

    s_init = np.sqrt(np.mean(np.sum((ref - ref.mean(0))**2, 1)) /
                     np.mean(np.sum((src - src.mean(0))**2, 1)))

    if center == 'nose':
        # Anchor nose tips (max-Z point in each cloud) instead of centroids.
        # More reliable for partial frontal scans whose centroid is biased.
        src_anchor = src[np.argmax(src[:, 2])]
        ref_anchor = ref[np.argmax(ref[:, 2])]
        src_c = (src - src_anchor) * s_init + ref_anchor
    else:
        src_c = (src - src.mean(0)) * s_init + ref.mean(0)
    s_total = s_init

    nn_ref = icp_ref.astype(np.float64) if icp_ref is not None else ref
    tree   = KDTree(nn_ref)

    for i in range(n_iter):
        dists, idx = tree.query(src_c, k=1)
        mask = dists < dist_thresh
        if mask.sum() < min_inliers:
            print(f'  iter {i+1}: only {mask.sum()} inliers — stopping early')
            break
        s_d, R, t = umeyama(src_c[mask], nn_ref[idx[mask]])
        src_c   = (s_d * R @ src_c.T).T + t
        s_total *= s_d
        dist_thresh *= thresh_decay
        print(f'  iter {i+1:2d}: s_total={s_total:.5f}  s_delta={s_d:.5f}  '
              f'inliers={mask.sum():5d}  thresh={dist_thresh:.4f}')
        if abs(s_d - 1.0) < 1e-5:
            print('  converged.')
            break

    return float(s_total), src_c.astype(np.float32)


# ── Pre-alignment for 2189: 180° around X axis ────────────────────────────────
R_flip_x = np.diag([1., -1., -1.])

# Front-face only ref for ICP: excludes back-of-head points.
ref_front = ref_pts[ref_pts[:, 2] >= np.median(ref_pts[:, 2])]
print(f'ref_front: {ref_front.shape[0]} pts  (Z >= {np.median(ref_pts[:,2]):.3f})')

CONFIGS = {
    'plank_scaled': dict(init_R=None),
    '2189':         dict(init_R=R_flip_x, icp_ref=ref_front, center='nose'),
}

results = {}
for name, src in SCANS.items():
    print(f'\n=== {name} ===')
    s_est, src_aligned = scale_icp(src, ref_pts, **CONFIGS[name])
    results[name] = {'s_est': s_est, 'src_aligned': src_aligned}
    model_pred = {'plank_scaled': 0.87, '2189': 0.607}[name]
    print(f'>>> {name}: s_est={s_est:.4f}   model_pred={model_pred}   '
          f'expected_pred={1/s_est:.4f}   bias={model_pred*s_est:.4f}')

ref_front: 5394 pts  (Z >= 0.226)

=== plank_scaled ===
  iter  1: s_total=1.27338  s_delta=0.93237  inliers= 2000  thresh=0.2250
  iter  2: s_total=1.22944  s_delta=0.96549  inliers= 1998  thresh=0.2025
  iter  3: s_total=1.20063  s_delta=0.97657  inliers= 1990  thresh=0.1823
  iter  4: s_total=1.17842  s_delta=0.98150  inliers= 1981  thresh=0.1640
  iter  5: s_total=1.15728  s_delta=0.98206  inliers= 1971  thresh=0.1476
  iter  6: s_total=1.13876  s_delta=0.98400  inliers= 1960  thresh=0.1329
  iter  7: s_total=1.12083  s_delta=0.98425  inliers= 1943  thresh=0.1196
  iter  8: s_total=1.10616  s_delta=0.98692  inliers= 1930  thresh=0.1076
  iter  9: s_total=1.09334  s_delta=0.98841  inliers= 1920  thresh=0.0969
  iter 10: s_total=1.08212  s_delta=0.98973  inliers= 1901  thresh=0.0872
  iter 11: s_total=1.07176  s_delta=0.99043  inliers= 1865  thresh=0.0785
  iter 12: s_total=1.06317  s_delta=0.99199  inliers= 1818  thresh=0.0706
  iter 13: s_total=1.05773  s_delta=0.99488  inliers= 17

In [33]:
# ── Global registration via FPFH + RANSAC ─────────────────────────────────────
# Needed when scan is in a different orientation from UHM canonical space (e.g. 2189).
# Returns (R, t) that roughly aligns src into ref's coordinate frame (no scale yet).

def make_o3d_pcd(pts, voxel=0.05):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd = pcd.voxel_down_sample(voxel)
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*4, max_nn=30))
    return pcd

def fpfh_features(pcd, voxel=0.05):
    return o3d.pipelines.registration.compute_fpfh_feature(
        pcd,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*10, max_nn=100)
    )

def global_registration(src_pts, ref_pts, voxel=0.05):
    """FPFH + RANSAC global registration. Returns 4×4 transform T (no scale)."""
    src_pcd  = make_o3d_pcd(src_pts, voxel)
    ref_pcd  = make_o3d_pcd(ref_pts, voxel)
    src_feat = fpfh_features(src_pcd, voxel)
    ref_feat = fpfh_features(ref_pcd, voxel)

    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        src_pcd, ref_pcd, src_feat, ref_feat,
        mutual_filter=True,
        max_correspondence_distance=voxel * 3,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel * 3),
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4_000_000, 0.999),
    )
    T = np.asarray(result.transformation)
    R, t = T[:3, :3], T[:3, 3]
    print(f'  RANSAC fitness={result.fitness:.4f}  inlier_rmse={result.inlier_rmse:.4f}')
    return R, t

In [34]:
# ── Model bias analysis ───────────────────────────────────────────────────────
model_preds = {'plank_scaled': 0.87, '2189': 0.607}

print('Scan             s_gt_est   pred_scale   bias(pred/gt)   error(%)')
print('-' * 65)
for name in SCANS:
    s_gt   = results[name]['s_est']
    s_pred = model_preds[name]
    bias   = s_pred / s_gt
    err_pct = (s_pred - s_gt) / s_gt * 100
    print(f'{name:<16}  {s_gt:.4f}     {s_pred:.4f}       {bias:.4f}          {err_pct:+.1f}%')

Scan             s_gt_est   pred_scale   bias(pred/gt)   error(%)
-----------------------------------------------------------------
plank_scaled      1.0368     0.8700       0.8391          -16.1%
2189              1.1701     0.6070       0.5187          -48.1%


In [35]:
# ── Visualise alignment quality ───────────────────────────────────────────────
def pcd_trace(pts, color, name, size=1.5, opacity=0.5):
    return go.Scatter3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )

for name, res in results.items():
    fig = go.Figure([
        pcd_trace(ref_pts,            'steelblue', 'ref',            size=1.0),
        pcd_trace(res['src_aligned'], 'tomato',    f'{name} aligned', size=2.0),
    ])
    fig.update_layout(
        title=f'{name}  —  scale ICP alignment  (s_est={res["s_est"]:.4f})',
        height=550,
        scene=dict(aspectmode='data'),
        margin=dict(l=0,r=0,b=0,t=40),
    )
    fig.show()